# Redrob Ranker Sandbox Demo
This notebook clones the repo, loads a small candidate sample, runs `rank.py` on CPU, writes `submission.csv`, and downloads it.
Use the bundled sample or upload `sample_candidates.json` when you want to test your own small file. For the official submission, use a 100-candidate file and set `TOP_K = 100`.
Colab link: https://colab.research.google.com/drive/1l_xRxQj8hkYATMzuDAzCwyWkaNIJ4rRi?usp=sharing

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/rangeshsha-Rookie/redrob-ranker"
WORKDIR = Path("/content/redrob-ranker")

if not (WORKDIR / "rank.py").exists():
    subprocess.run(["git", "clone", REPO_URL, str(WORKDIR)], check=True)

os.chdir(WORKDIR)
print("Working directory:", Path.cwd())
print("Repo files ready:", (Path("rank.py").exists()))

In [ ]:
import subprocess
import sys
from pathlib import Path

requirements = Path("requirements.txt")
if requirements.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)], check=True)
else:
    print("No requirements.txt found, so nothing extra is installed.")

In [ ]:
import json
from pathlib import Path

def load_candidates_file() -> Path:
    local_sample = Path("sample_candidates.json")
    if local_sample.exists():
        return local_sample

    from google.colab import files

    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("Upload sample_candidates.json to continue.")
    return Path(next(iter(uploaded)))

def ensure_jsonl(path: Path) -> Path:
    raw = path.read_text(encoding="utf-8").strip()
    if not raw:
        raise ValueError("Candidate file is empty.")
    if raw[0] == "[":
        items = json.loads(raw)
        jsonl_path = path.with_suffix(".jsonl")
        with jsonl_path.open("w", encoding="utf-8", newline="\n") as handle:
            for item in items:
                handle.write(json.dumps(item, ensure_ascii=False) + "\n")
        return jsonl_path
    return path

candidate_file = load_candidates_file()
candidate_jsonl = ensure_jsonl(candidate_file)
candidate_rows = [line for line in candidate_jsonl.read_text(encoding="utf-8").splitlines() if line.strip()]
candidate_count = len(candidate_rows)
top_k = min(100, candidate_count)

print("Input file:", candidate_jsonl)
print("Candidate count:", candidate_count)
print("Top-K used:", top_k)
if candidate_count < 100:
    print("Demo mode: fewer than 100 candidates were provided, so the notebook ranks all available rows.")

In [ ]:
import csv
import subprocess
import sys
from pathlib import Path

out_path = Path("submission.csv")
subprocess.run([
    sys.executable,
    "rank.py",
    "--candidates",
    str(candidate_jsonl),
    "--out",
    str(out_path),
    "--top-k",
    str(top_k),
], check=True)

with out_path.open("r", encoding="utf-8", newline="") as handle:
    rows = list(csv.reader(handle))

print("Wrote:", out_path)
print("Output rows including header:", len(rows))
print("Header:", rows[0])
print("First ranked row:", rows[1])

In [ ]:
from google.colab import files

files.download("submission.csv")